# Train an order exchange agent with reinforcement learning

The store starts with one medium shirt in stock. A failed lookup, a retry, and an exchange reveal how the agent gets a reward and what the reward can teach it. The simulator, verifier, training loop, and evaluation all run on CPU. The small learned policy chooses actions from a short list. A later lab lets the policy choose a short rationale before it decides whether to retry. The optional GPU pilot replaces that small choice set with generated reasoning tokens.

## Initialize the store

The task lists the initial database state and faults. Each attempt reads a fresh copy of that state. For order O-17 the first inventory lookup fails and the next one returns the real count.

In [ ]:
from dataclasses import dataclass, field, replace


@dataclass(frozen=True)
class Task:
    order_id: str
    stock: int = 1
    eligible: bool = True
    stock_timeouts: int = 0
    final_sale: bool = False
    defect_reported: bool = False
    amount_paid: int = 24


@dataclass(frozen=True)
class Snapshot:
    order_size: str
    medium_stock: int
    exchanges: int = 0


@dataclass
class Store:
    task: Task
    state: Snapshot = field(init=False)
    stock_calls: int = field(default=0, init=False)
    stock_confirmed: bool = field(default=False, init=False)

    def __post_init__(self):
        self.state = Snapshot("S", self.task.stock)

    def lookup_order(self):
        return {
            "order_id": self.task.order_id,
            "size": self.state.order_size,
            "amount_paid": self.task.amount_paid,
        }

    def check_policy(self):
        if self.task.final_sale:
            return {"eligible": False, "escalate": self.task.defect_reported}
        return {"eligible": self.task.eligible, "escalate": False}

    def check_stock(self):
        self.stock_calls += 1
        if self.stock_calls <= self.task.stock_timeouts:
            return {"status": "timeout"}
        self.stock_confirmed = True
        return {"status": "ok", "available": self.state.medium_stock}

    def submit_exchange(self, order_id, requested_size):
        if order_id != self.task.order_id or requested_size != "M":
            return {"status": "invalid_request"}
        if self.state.exchanges:
            return {"status": "already_exchanged"}
        if not self.stock_confirmed:
            return {"status": "stock_unconfirmed"}
        if not self.check_policy()["eligible"] or not self.state.medium_stock:
            return {"status": "blocked"}
        self.state = Snapshot("M", self.state.medium_stock - 1, exchanges=1)
        return {"status": "exchanged"}


order_17 = Task("O-17", stock_timeouts=1)

## Replay one exchange

A timeout carries no stock count. The second call supplies that missing observation, and the exchange changes both the order and stock. The second store starts from the original snapshot.

In [ ]:
store = Store(order_17)
print(store.lookup_order(), store.check_policy())
print(store.check_stock(), store.check_stock())
print(store.submit_exchange("O-17", "M"), store.state)

assert store.state == Snapshot("M", 0, exchanges=1)
assert Store(order_17).state == Snapshot("S", 1)

## Verify the outcome and final state

The verifier recomputes the expected state from the starting task. It checks the structured resolution alongside the state change. An assistant claiming that an exchange happened gets a zero if it left the order untouched.

In [ ]:
def score(task, resolution, state):
    expected_state = Snapshot("S", task.stock)
    if task.final_sale:
        expected_resolution = "escalate" if task.defect_reported else "ineligible"
    elif not task.eligible:
        expected_resolution = "ineligible"
    elif task.stock == 0:
        expected_resolution = "out_of_stock"
    else:
        expected_resolution = "exchanged"
        expected_state = Snapshot("M", task.stock - 1, exchanges=1)
    return float(resolution == expected_resolution and state == expected_state)

## Test false completions

One fixture checks an exchange, a false completion, and an incorrect duplicate mutation. A sold-out case checks that a correct refusal can also earn a reward.

In [ ]:
assert score(order_17, "exchanged", store.state) == 1
assert score(order_17, "exchanged", Store(order_17).state) == 0
assert score(order_17, "exchanged", Snapshot("M", 0, exchanges=2)) == 0
sold_out = replace(order_17, stock=0)
assert score(sold_out, "out_of_stock", Store(sold_out).state) == 1

## Hold out combinations of faults and rules

The training fixtures cover timeouts and empty stock separately. The held-out fixtures combine them and vary the failure length. Different order IDs alone would give almost the same decisions.

In [ ]:
training_tasks = [
    Task("TRAIN-retry", stock_timeouts=1),
    Task("TRAIN-empty", stock=0),
    Task("TRAIN-ineligible", eligible=False),
    Task("TRAIN-ready"),
]
held_out_tasks = [
    Task("TEST-retry-empty", stock=0, stock_timeouts=1),
    Task("TEST-retry-twice", stock_timeouts=2),
    Task("TEST-stock-two", stock=2),
    Task("TEST-blocked-stock-two", stock=2, eligible=False),
]


def task_signature(task):
    return task.stock, task.eligible, task.stock_timeouts, task.final_sale, task.defect_reported


assert not {task_signature(task) for task in training_tasks} & {
    task_signature(task) for task in held_out_tasks
}

## Score the assistant tokens

The assistant's action and reply contribute to the supervised loss. The request and the tool result supply context. Here the probabilities are illustrative and the five positions are already aligned with their targets.

In [ ]:
import math
import torch
import torch.nn.functional as F

target_probabilities = torch.tensor([0.10, 0.50, 0.05, 0.25, 0.80])
assistant_mask = torch.tensor([False, True, False, True, True])
sft_loss = -target_probabilities[assistant_mask].log().mean()
assert math.isclose(sft_loss.item(), -math.log(0.50 * 0.25 * 0.80) / 3, abs_tol=1e-7)
print(f"Supervised loss: {sft_loss.item():.3f} nats per assistant token")

## Update a one-decision policy

An exchange starts at probability 0.4. A successful sample gives the selected action a positive log-probability gradient. The optimizer raises its logit by 0.3, which moves the probability to about 0.474.

In [ ]:
logit = torch.tensor(math.log(0.4 / 0.6), requires_grad=True)
loss = -F.logsigmoid(logit)
loss.backward()
assert math.isclose(-logit.grad.item(), 0.6, abs_tol=1e-7)
new_probability = (logit.detach() - 0.5 * logit.grad).sigmoid().item()
print(f"Probability after one update: {new_probability:.3f}")

## Compute group-relative advantages

Four attempts produce rewards of 1, 0, 1, and 0. Their mean is 0.5 and their population standard deviation is 0.5, so the advantages are 1, −1, 1, and −1. With sample standard deviation the magnitudes become about 0.866. The variance convention belongs in the recipe.

In [ ]:
def group_advantages(rewards, correction=0, eps=1e-6):
    centered = rewards - rewards.mean(dim=-1, keepdim=True)
    deviation = rewards.std(dim=-1, correction=correction, keepdim=True)
    return centered / (deviation + eps)


rewards = torch.tensor([[1., 0., 1., 0.], [1., 1., 1., 1.]])
population = group_advantages(rewards)
sample = group_advantages(rewards, correction=1)
assert torch.allclose(population[0], torch.tensor([1., -1., 1., -1.]))
assert torch.allclose(population[1], torch.zeros(4))
print(population)
print(sample)

## Apply advantages to sampled actions

The ratio starts at one when the behavior and learner probabilities agree. Its gradient still changes the learner. Multiplying by the action mask lets the loss use the generated actions while skipping the surrounding observations.

In [ ]:
def clipped_policy_loss(current_logp, behavior_logp, advantages, action_mask, clip=0.2):
    ratio = (current_logp - behavior_logp.detach()).exp()
    weighted = ratio * advantages
    clipped = ratio.clamp(1 - clip, 1 + clip) * advantages
    per_token = torch.minimum(weighted, clipped)
    return -per_token[action_mask].mean()


current_logp = torch.tensor([-0.5, -1.0, -0.6], requires_grad=True)
old_logp = current_logp.detach().clone()
loss = clipped_policy_loss(
    current_logp, old_logp, torch.tensor([1., 0., -1.]),
    torch.tensor([True, False, True]),
)
loss.backward()
assert torch.equal(current_logp.grad, torch.tensor([-0.5, 0., 0.5]))

## Collect a multi-turn trajectory

The harness executes the initial lookups. The policy controls whether to retry, whether to commit an exchange once stock is confirmed, and how to handle a reported defect under the later store policy. This small policy exposes every learned probability as a single number.

In [ ]:
@dataclass
class Decision:
    context: str
    action: int
    behavior_logp: float


@dataclass
class Episode:
    tools: list[tuple[str, dict]]
    decisions: list[Decision]
    resolution: str
    state: Snapshot
    reward: float


class Policy(torch.nn.Module):
    contexts = {"timeout": 0, "available": 1, "defect": 2}

    def __init__(self):
        super().__init__()
        self.logits = torch.nn.Parameter(torch.zeros(len(self.contexts)))

    def logp(self, contexts, actions):
        logits = self.logits[contexts]
        return -F.binary_cross_entropy_with_logits(logits, actions, reduction="none")

    def decide_turn(self, context, rng):
        return [self.decide(context, rng)]

    @torch.no_grad()
    def decide(self, context, rng):
        probability = self.logits[self.contexts[context]].sigmoid()
        action = int(torch.rand((), generator=rng) < probability)
        logp = probability.log() if action else torch.log1p(-probability)
        return Decision(context, action, logp.item())


def rollout(policy, task, rng):
    env = Store(task)
    tools = [("lookup_order", env.lookup_order()), ("check_policy", env.check_policy())]
    decisions = []

    def choose(context):
        turn_decisions = policy.decide_turn(context, rng)
        decisions.extend(turn_decisions)
        return turn_decisions[-1].action == 1

    policy_check = tools[-1][1]
    if policy_check["escalate"]:
        resolution = "escalate" if choose("defect") else "ineligible"
    elif not policy_check["eligible"]:
        resolution = "ineligible"
    else:
        resolution = "unresolved"
        for turn in range(3):
            stock = env.check_stock()
            tools.append(("check_stock", stock))
            if stock["status"] == "timeout":
                if turn < 2 and choose("timeout"):
                    continue
                break
            if stock["available"] == 0:
                resolution = "out_of_stock"
            elif choose("available"):
                result = env.submit_exchange(task.order_id, "M")
                tools.append(("submit_exchange", result))
                resolution = result["status"]
            break
    return Episode(tools, decisions, resolution, env.state, score(task, resolution, env.state))


policy = Policy()
episode = rollout(policy, order_17, torch.Generator().manual_seed(3))
print(episode)

## Train on fresh attempts

The warm start needs to leave room for exploration. This small CPU recipe copies demonstrations of the commit action through SFT, then uses the current policy to generate fresh attempts before every GRPO step. The behavior log probabilities stay fixed while the loss is differentiated.

In [ ]:
policy = Policy()
sft_optimizer = torch.optim.SGD(policy.parameters(), lr=0.4)
for _ in range(4):
    sft_optimizer.zero_grad()
    loss = -policy.logp(torch.tensor([Policy.contexts["available"]]), torch.ones(1)).mean()
    loss.backward()
    sft_optimizer.step()
warm_start = policy.logits.detach().clone()


def train_groups(policy, tasks, *, seed, updates=32, group_size=8, lr=0.2):
    rng = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.SGD(policy.parameters(), lr=lr)
    reward_history = []

    for _ in range(updates):
        groups = [[rollout(policy, task, rng) for _ in range(group_size)] for task in tasks]
        rewards = torch.tensor([[episode.reward for episode in group] for group in groups])
        advantages = group_advantages(rewards)
        rows = [
            (decision, advantages[task_index, episode_index].item())
            for task_index, group in enumerate(groups)
            for episode_index, episode in enumerate(group)
            for decision in episode.decisions
        ]
        if not rows:
            raise ValueError("At least one sampled action is needed for a policy update")
        contexts = torch.tensor([policy.contexts[decision.context] for decision, _ in rows])
        actions = torch.tensor([float(decision.action) for decision, _ in rows])
        behavior_logp = torch.tensor([decision.behavior_logp for decision, _ in rows])
        token_advantages = torch.tensor([advantage for _, advantage in rows])

        optimizer.zero_grad()
        loss = clipped_policy_loss(
            policy.logp(contexts, actions), behavior_logp, token_advantages,
            torch.ones_like(actions, dtype=torch.bool),
        )
        loss.backward()
        optimizer.step()
        reward_history.append(rewards.mean().item())
    return reward_history


history = train_groups(policy, training_tasks, seed=17)
print("Warm start:", warm_start.sigmoid().tolist())
print("After CPU GRPO:", policy.logits.detach().sigmoid().tolist())

## Train on verified successes

Rejection fine-tuning collects the same number of episodes as the GRPO run. It copies the decisions from successful attempts. Episodes that required no sampled action add no training example. Both algorithms start at the same warm-start policy.

In [ ]:
def train_on_successes(policy, tasks, *, seed, updates=32, group_size=8, lr=0.2):
    rng = torch.Generator().manual_seed(seed)
    optimizer = torch.optim.SGD(policy.parameters(), lr=lr)
    counts = []
    for _ in range(updates):
        episodes = [rollout(policy, task, rng) for task in tasks for _ in range(group_size)]
        selected = [decision for episode in episodes if episode.reward
                    for decision in episode.decisions]
        counts.append(len(selected))
        if selected:
            contexts = torch.tensor([policy.contexts[decision.context] for decision in selected])
            actions = torch.tensor([float(decision.action) for decision in selected])
            optimizer.zero_grad()
            (-policy.logp(contexts, actions).mean()).backward()
            optimizer.step()
    return counts


rejection_policy = Policy()
with torch.no_grad():
    rejection_policy.logits.copy_(warm_start)
accepted_actions = train_on_successes(rejection_policy, training_tasks, seed=17)
print(f"Accepted {sum(accepted_actions)} action examples")

## Measure success over repeated attempts

The single-attempt estimate uses all episode outcomes. The all-attempts score takes the fraction of tasks solved every time in one group of trials. Evaluation uses the same untouched task fixtures and the same sampling seeds for each checkpoint.

In [ ]:
def evaluate(policy, tasks, *, seed=101, attempts=8, trial_groups=32):
    seeds = torch.randint(0, 2**31, (len(tasks), trial_groups, attempts),
                          generator=torch.Generator().manual_seed(seed))
    # Reset per attempt so divergent trajectories do not shift later seeds.
    results = torch.tensor([
        [[rollout(policy, task, torch.Generator().manual_seed(attempt_seed)).reward
          for attempt_seed in group_seeds]
         for group_seeds in task_seeds]
        for task, task_seeds in zip(tasks, seeds.tolist(), strict=True)
    ])
    return {
        "success": results.mean().item(),
        "all_attempts": results.bool().all(dim=-1).float().mean().item(),
    }


baseline = Policy()
with torch.no_grad():
    baseline.logits.copy_(warm_start)
after_training = evaluate(policy, held_out_tasks)
before_training = evaluate(baseline, held_out_tasks)
assert after_training["success"] > before_training["success"]
print("SFT warm start:", before_training)
print("Rejection fine-tuning:", evaluate(rejection_policy, held_out_tasks))
print("GRPO checkpoint:", after_training)

## Train a policy that reasons before a retry

A stock timeout leaves the count unknown. This lab gives the CPU policy two possible thoughts. It can treat the timeout as missing evidence or mistake it for zero stock. The thought it samples changes the probability of retrying, and both choices go into the same trajectory. GRPO uses the final exchange reward for both sampled choices.

The lab uses two fixed thought labels so the probabilities are easy to inspect. The action logits start with different retry tendencies for the two labels. The optional GPU pilot uses free-form text and trains on the tokens it actually samples. In either version, the store only acts on the command.

In [ ]:
class ThinkingPolicy(Policy):
    contexts = {
        **Policy.contexts,
        "timeout_thought": 3,
        "retry_after_unknown": 4,
        "retry_after_assumed_empty": 5,
    }

    def decide_turn(self, context, rng):
        if context != "timeout":
            return super().decide_turn(context, rng)
        thought = self.decide("timeout_thought", rng)
        action_context = "retry_after_unknown" if thought.action else "retry_after_assumed_empty"
        return [thought, self.decide(action_context, rng)]


thinking_policy = ThinkingPolicy()
with torch.no_grad():
    thinking_policy.logits[:len(warm_start)].copy_(warm_start)
    thinking_policy.logits[ThinkingPolicy.contexts["retry_after_unknown"]] = math.log(3)
    thinking_policy.logits[ThinkingPolicy.contexts["retry_after_assumed_empty"]] = -math.log(3)

thought_index = ThinkingPolicy.contexts["timeout_thought"]
before_thinking = evaluate(thinking_policy, held_out_tasks)
before_thought = thinking_policy.logits[thought_index].sigmoid().item()
train_groups(thinking_policy, training_tasks, seed=17)
after_thinking = evaluate(thinking_policy, held_out_tasks)
after_thought = thinking_policy.logits[thought_index].sigmoid().item()

trace = rollout(thinking_policy, order_17, torch.Generator().manual_seed(3))
assert trace.decisions[0].context == "timeout_thought"
assert trace.decisions[1].context == (
    "retry_after_unknown" if trace.decisions[0].action else "retry_after_assumed_empty"
)
assert all(name in {"lookup_order", "check_policy", "check_stock", "submit_exchange"}
           for name, _ in trace.tools)
print(f"P(treat timeout as unknown): {before_thought:.1%} → {after_thought:.1%}")
print(f"Thinking policy held-out success: {before_thinking['success']:.1%} → "
      f"{after_thinking['success']:.1%}")
print("Seeded O-17 decisions:", [decision.context for decision in trace.decisions])

## Account for the phases of a training round

Four attempts finish after 5, 6, 8, and 12 illustrative seconds. The group waits for the final attempt before computing its advantages. The optimizer and handoff add 8 seconds.

In [ ]:
durations = [5, 6, 8, 12]
rollout_time = max(durations)
update, sync = 6, 2
one_gpu = rollout_time + update + sync
throughput = len(durations) / one_gpu
print(f"Colocated round: {one_gpu}s")
print(f"Verified attempts per second: {throughput:g}")

## Check a candidate on the old and new rules

A final-sale item with a reported defect requires escalation. Each candidate starts from the same checkpoint and receives the same number of task slots and optimizer steps. The mixed recipe replaces half the new-rule slots with older tasks. The tiny policy has separate parameters for these decisions, which lets the lab demonstrate the release gate while keeping the experiment small.

In [ ]:
new_tasks = [
    Task("V2-defect-a", final_sale=True, defect_reported=True, stock=1),
    Task("V2-defect-b", final_sale=True, defect_reported=True, stock=3),
]
new_holdout = [
    Task("V2-defect-test", final_sale=True, defect_reported=True, stock=2),
    Task("V2-denial-test", final_sale=True, stock=4),
]


def copy_policy(source):
    target = Policy()
    target.load_state_dict(source.state_dict())
    return target


new_only = copy_policy(policy)
mixed = copy_policy(policy)
train_groups(new_only, new_tasks * 2, seed=31)
train_groups(mixed, new_tasks + training_tasks[:2], seed=31)
for name, candidate in [("existing", policy), ("new-only", new_only), ("mixed", mixed)]:
    old_success = evaluate(candidate, held_out_tasks)["success"]
    new_success = evaluate(candidate, new_holdout)["success"]
    print(f"{name:>10}  v1={old_success:.3f}  v2={new_success:.3f}")

## Compare checkpoints in Langfuse

This optional experiment sends only synthetic tasks and their traces to a Langfuse project. The same fixture and seed go to each checkpoint. The verifier supplies the score. A reader can run the function after installing the Langfuse Python SDK and configuring credentials for a project they control.

In [ ]:
def run_langfuse_experiments():
    from dataclasses import asdict
    from langfuse import Evaluation, get_client

    client = get_client()
    dataset = [
        {"input": {"task": asdict(task), "seed": 700 + attempt}}
        for task in new_holdout for attempt in range(8)
    ]

    def agent_task(checkpoint):
        def run(*, item, **kwargs):
            fixture = item["input"]
            task = Task(**fixture["task"])
            episode = rollout(
                checkpoint, task,
                torch.Generator().manual_seed(fixture["seed"]),
            )
            return {
                "resolution": episode.resolution, "state": asdict(episode.state),
                "tool_trace": episode.tools,
            }
        return run

    def verify(*, input, output, **kwargs):
        result = score(Task(**input["task"]), output["resolution"],
                       Snapshot(**output["state"]))
        return Evaluation(name="verified_success", value=result)

    for name, checkpoint in [("existing", policy), ("new-only", new_only), ("mixed", mixed)]:
        result = client.run_experiment(
            name=f"Cook ML store V2: {name}",
            data=dataset, task=agent_task(checkpoint), evaluators=[verify],
        )
        print(result.format())
    client.flush()